# 01 · First Contact With a Dataset

The first fifteen minutes with unfamiliar data decide how much time you waste later.

This notebook is deliberately not a cleaning tutorial. It is the **survey** you do
before deciding anything — what's here, what shape it's in, what's obviously wrong.
Resist the urge to fix anything yet.

**Before you start:** open `artifacts/project-intake.md` and answer question 3
(*what data exists?*) from memory. Then check yourself against what you find below.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

DATA = Path.cwd()
while not (DATA / "data" / "turbine_readings.csv").exists() and DATA != DATA.parent:
    DATA = DATA.parent
CSV = DATA / "data" / "turbine_readings.csv"
print("reading", CSV)

reading /Applications/Development/Projects/pgp-aiml/data/turbine_readings.csv


In [2]:
df = pd.read_csv(CSV)
df.shape

(6150, 11)

## The four commands you always run

`shape`, `head`, `dtypes`, `describe`. Nothing clever — but read the output properly
rather than glancing at it.

In [3]:
df.head()

,turbine_id,site,install_date,age_years,rotor_speed_rpm,vibration_mm_s,gearbox_temp,operating_hours,last_inspector,failed_within_90d,maintenance_ticket_raised
0,T00567,Clyde,2019-05-12,7.23,11.99,0.5371,180.5,"48,806",s.tran,0,routine
1,T02105,Ardrossan,2025-02-22,1.44,11.29,NaN,57.2,"9,944",m_lindqvist,0,routine
2,T04536,Beinn an Tuirc,2024-07-18,2.04,13.44,0.1917,151.5,"14,513",M Lindqvist,0,NaN
3,T02556,Whitelee,2020-08-17,5.96,11.67,0.2849,61.3,"40,876",s.tran,1,Urgent
4,T04418,Clyde,2023-06-09,3.15,13.72,0.3076,154.4,"20,349",m_lindqvist,0,NaN


In [4]:
df.dtypes

turbine_id                    object
site                          object
install_date                  object
age_years                    float64
rotor_speed_rpm              float64
vibration_mm_s               float64
gearbox_temp                 float64
operating_hours               object
last_inspector                object
failed_within_90d              int64
maintenance_ticket_raised     object
dtype: object

### Read those dtypes again

`operating_hours` came back as `object`, not a number. That means pandas found
something non-numeric in the column and gave up on parsing it. **A numeric column
that loads as `object` is always worth investigating** — it usually means the data
has text in it somewhere.

Let's see what.

In [5]:
print(df["operating_hours"].dropna().head(8).tolist())
print()
# rows that won't convert to a number
bad = pd.to_numeric(df["operating_hours"].str.replace(",", ""), errors="coerce").isna() \
      & df["operating_hours"].notna()
print(f"{bad.sum()} values contain something non-numeric:")
print(df.loc[bad, "operating_hours"].head(5).tolist())

['48,806', '9,944', '14,513', '40,876', '20,349', '39,438', '11,937', '18,303']

1 values contain something non-numeric:
['36,300 hrs']


Thousands separators, and some values carry a `hrs` suffix. Two different
formatting problems in one column — completely normal for data assembled by hand
or exported from more than one system.

## Missingness

How much is missing, and *where*.

In [6]:
miss = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct": (df.isna().mean() * 100).round(1),
}).query("missing > 0").sort_values("pct", ascending=False)
miss

,missing,pct
maintenance_ticket_raised,4828,78.5
last_inspector,497,8.1
vibration_mm_s,418,6.8
rotor_speed_rpm,387,6.3
operating_hours,388,6.3


## Duplicates

Two kinds, and they need different treatment: identical rows, and the same real-world
thing recorded twice under different identifiers.

In [7]:
print(f"exact duplicate rows : {df.duplicated().sum()}")
print(f"duplicate turbine_id  : {df['turbine_id'].duplicated().sum()}")
print(f"unique turbines       : {df['turbine_id'].nunique()} of {len(df)} rows")

exact duplicate rows : 66
duplicate turbine_id  : 90
unique turbines       : 6060 of 6150 rows


The exact duplicates are easy. But are there turbines logged twice under
*different* ids? Check for rows identical in everything except the identifier.

In [8]:
cols = [c for c in df.columns if c != "turbine_id"]
relogged = df.duplicated(subset=cols, keep=False) & ~df.duplicated(keep=False)
print(f"{relogged.sum()} rows look like the same turbine under a different id")
df.loc[relogged].sort_values("install_date").head(4)

98 rows look like the same turbine under a different id


,turbine_id,site,install_date,age_years,rotor_speed_rpm,vibration_mm_s,gearbox_temp,operating_hours,last_inspector,failed_within_90d,maintenance_ticket_raised
5992,T00372,Whitelee,2015-10-24,10.78,14.31,0.4722,72.8,NaN,S. Tran,0,NaN
1697,T90050,Whitelee,2015-10-24,10.78,14.31,0.4722,72.8,NaN,S. Tran,0,NaN
3479,T90052,Clyde,2017-10-08,8.82,19.35,0.5806,164.8,"63,381",S. Tran,0,NaN
5186,T00674,Clyde,2017-10-08,8.82,19.35,0.5806,164.8,"63,381",S. Tran,0,NaN


## The target

Always look at the target's balance before anything else. It determines which
metrics mean anything at all.

In [9]:
rate = df["failed_within_90d"].mean()
print(f"failure rate: {rate:.2%}")
print(f"always-predict-'no-failure' accuracy: {1 - rate:.2%}   <- the number to beat")

failure rate: 15.38%
always-predict-'no-failure' accuracy: 84.62%   <- the number to beat


Write that baseline down. Any accuracy figure you produce later is meaningless
without it — this is the intake sheet's question 2, and on imbalanced data it is the
whole interpretation.

## Categorical columns

In [10]:
for c in ["site", "last_inspector", "maintenance_ticket_raised"]:
    print(f"--- {c}  ({df[c].nunique()} distinct)")
    print(df[c].value_counts(dropna=False).head(8))
    print()

--- site  (4 distinct)
site
Ardrossan         2194
Whitelee          1793
Clyde             1250
Beinn an Tuirc     913
Name: count, dtype: int64

--- last_inspector  (6 distinct)
last_inspector
s.tran         1201
j.okafor       1178
m_lindqvist    1142
J. Okafor       735
S. Tran         711
M Lindqvist     686
NaN             497
Name: count, dtype: int64

--- maintenance_ticket_raised  (4 distinct)
maintenance_ticket_raised
NaN        4828
routine     376
URGENT      332
urgent      319
Urgent      295
Name: count, dtype: int64



### Two things just surfaced

1. **`last_inspector` has the same people entered several ways** — `j.okafor` and
   `J. Okafor` are one person. Distinct counts are inflated and any grouping is wrong.

2. **`maintenance_ticket_raised` deserves suspicion.** Ask the intake sheet's
   question: *would I have this value, with this content, at the moment I need the
   prediction?* A maintenance ticket is raised **because** something failed. Check it.

In [11]:
pd.crosstab(df["maintenance_ticket_raised"].fillna("(none)"),
            df["failed_within_90d"], normalize="index").round(3)

failed_within_90d,0,1
maintenance_ticket_raised,,
(none),1.0,0.0
URGENT,0.0,1.0
Urgent,0.0,1.0
routine,1.0,0.0
urgent,0.0,1.0


Every `URGENT` row is a failure. That column is a **leaking feature** — it
encodes the answer and would not exist at prediction time. Train on it and you'll see
a superb score and a worthless model.

Found in the first fifteen minutes, because we asked. That is the entire point of
this pass.

---

## Your turn

1. `gearbox_temp` has a range from about 30 to 234. Plot it grouped by `site`.
   What do you notice, and what would it do to a model trained on this column?
2. `rotor_speed_rpm` has a few values of exactly 9999.0. What are they, and why
   is that specific number a giveaway?
3. Fill in question 3 of the intake sheet properly now.

Notebook 02 takes these findings and does something about them.